# 图 3-1 · 细菌形态分类
在项目目录启动 Jupyter，按顺序运行全部单元格即可生成 `bacteria.owl`。重新运行会覆盖同名文件，请将 Protégé 中的手工修改另存。模型表达图中的简化分类。

In [1]:
from pathlib import Path
from owlready2 import (
    AllDisjoint, DataProperty, FunctionalProperty, Not, ObjectProperty, Thing, World,
)

world = World()
onto = world.get_ontology("https://example.org/ontologies/bacteria.owl")
output = Path("bacteria.owl").resolve()

## 类层次
保留下方类图的直接父类，并补入上表提到的 Salmonella。Bacillus 在本图中指杆状细菌。

In [2]:
with onto:
    class Bacterium(Thing): pass
    class Coccus(Bacterium): pass
    class Bacillus(Bacterium): pass
    class Streptococcus(Bacterium): pass
    class Staphylococcus(Bacterium): pass
    class Pseudomonas(Bacterium): pass
    class Salmonella(Bacterium): pass

    class Shape(Thing): pass
    class Round(Shape): pass
    class Elongated(Shape): pass

    class Grouping(Thing): pass
    class Isolated(Grouping): pass
    class InPair(Grouping): pass
    class InCluster(Grouping): pass
    class InChain(Grouping): pass
    class InSmallChain(InChain): pass
    class InLongChain(InChain): pass

    AllDisjoint([Bacterium, Shape, Grouping])
    AllDisjoint([Round, Elongated])

## 属性与多重性
每个细菌恰有一个形状，可以有零到多个排列。革兰氏染色结果和菌落数分别用布尔值和整数表示。

In [3]:
with onto:
    class has_shape(ObjectProperty):
        domain = [Bacterium]
        range = [Shape]

    class has_grouping(ObjectProperty):
        domain = [Bacterium]
        range = [Grouping]

    class gram_positive(DataProperty, FunctionalProperty):
        domain = [Bacterium]
        range = [bool]

    class nb_colonies(DataProperty, FunctionalProperty):
        domain = [Bacterium]
        range = [int]

    Bacterium.is_a.append(has_shape.exactly(1, Shape))

## 形态和染色特征
球菌和杆菌用形状定义，因此推理器可以按形状归类。属名只附上图示的必要条件。表中两种革兰氏阴性杆菌共用一个跨行单元格，保留单个或成对的可能性。空白单元格不产生排除规则。

In [4]:
with onto:
    Coccus.equivalent_to.append(Bacterium & has_shape.some(Round))
    Bacillus.equivalent_to.append(Bacterium & has_shape.some(Elongated))

    Staphylococcus.is_a.extend([
        has_shape.some(Round), has_grouping.some(InCluster), gram_positive.value(True),
    ])
    Streptococcus.is_a.extend([
        has_shape.some(Round), has_grouping.some(InChain),
        Not(has_grouping.some(Isolated)), gram_positive.value(True),
    ])
    for genus in (Pseudomonas, Salmonella):
        genus.is_a.extend([
            has_shape.some(Elongated), has_grouping.some(Isolated | InPair),
            gram_positive.value(False),
        ])

## 导出并回读
以 Protégé 可打开的 RDF/XML 格式保存，并在独立的 Owlready2 World 中检查导出内容。

In [5]:
onto.save(file=str(output), format="rdfxml")
loaded = World().get_ontology(output.as_uri()).load()
assert {c.name for c in loaded.classes()} == {c.name for c in onto.classes()}
assert {p.name for p in loaded.object_properties()} == {"has_shape", "has_grouping"}
assert {p.name for p in loaded.data_properties()} == {"gram_positive", "nb_colonies"}
assert loaded.InChain in loaded.InSmallChain.is_a and loaded.InChain in loaded.InLongChain.is_a
assert loaded.has_shape.exactly(1, loaded.Shape) in loaded.Bacterium.is_a
assert Not(loaded.has_grouping.some(loaded.Isolated)) in loaded.Streptococcus.is_a
print(f"已导出 {output.name}，共 {len(list(loaded.classes()))} 个命名类。")

已导出 bacteria.owl，共 17 个命名类。
